# Jugalbandi — GRPO Training (Colab)

Fine-tunes Qwen2.5-0.5B-Instruct with GRPO against the Jugalbandi OpenEnv reward signal.

See `docs/Train.md` for the full walkthrough of every decision made here, and
`docs/EXPERIMENT_PLAN.md` for what this training run is actually being evaluated
against once it's done.

Training is **in-process and per-step** (updatedplan.md Phase 3, revised for Claim
B — in-context regime inference from reward feedback alone, §9 decision 2,
resolved in favour of Claim B): each GRPO sample is one decision point sampled
from a reference episode (which contains a drift event, or, for a 20% control
condition, no drift at all). The prompt shows the outcome of the *previous* step
(`raaga_env.prompting.StepFeedback`, Phase 2.3) — this is what gives an
otherwise-blind arm (DIAL/HIDDEN) any chance of noticing a rule change from
reward correlation alone, since without it, reward is never in the context at
all. The completion is one action; the reward is that action's immediate reward
plus a short Monte-Carlo continuation under a reference policy, computed by
restoring the exact env state the prompt was drawn from
(`JugalbandiEnv.set_state`, Phase 0.4) and replaying via
`eval.rollout.rollout_from_state` — the same function every baseline and
evaluation number goes through, so training reward and eval reward can never
silently diverge.

An earlier version of this notebook generated a full blind 64-action episode per
completion. That design can't use per-step feedback (a completion generated in
one shot can't react to feedback about steps it hasn't taken yet), so it was
replaced once Claim B was chosen — see `training/train_grpo.py`'s module
docstring for the full account.

This notebook is a thin driver over `training/train_grpo.py` — the pipeline logic
(dataset construction, drift scheduling, reward scoring) lives there and is
imported, not redefined here (Phase 3.4). Run cells top to bottom.

## Cell 1 — Install Dependencies

In [ ]:
# Takes ~3 minutes on Colab.
import os
from google.colab import userdata

# The repo must be present locally: raaga_env.prompting is the single
# source of truth for every prompt this notebook builds (updatedplan.md
# Phase 2.1), and requirements-train.txt lives in it too.
#
# This repo is PRIVATE, so cloning needs a token. Create a fine-grained
# GitHub PAT scoped to read-only access on just this repo
# (github.com/settings/personal-access-tokens/new -> Repository access:
# "Only select repositories" -> jayyyyqwq/raga -> Permissions: Contents:
# Read-only), then add it as a Colab secret named GH_TOKEN (key icon in the
# left sidebar). The token never gets written into this notebook file.
GH_TOKEN = userdata.get("GH_TOKEN")
REPO_URL = f"https://{GH_TOKEN}@github.com/jayyyyqwq/raga.git"
REPO_DIR = "raga"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

# unsloth installs from git (its own recommendation, for hardware-specific
# kernels); everything else is pinned to match requirements-train.txt so the
# Colab run and any local re-scoring use the same dependency graph. Training
# is in-process (Phase 3.1), so no HTTP client is needed here.
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -r requirements-train.txt gymnasium numpy

## Cell 2 — Mount Drive + Set Secrets

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")   # set in Colab Secrets
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")             # for pushing model at end

## Cell 3 — Load Model

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B-Instruct",
    max_seq_length=512,  # one action per completion again — no need for episode-mode's headroom
    load_in_4bit=True,
    token=os.environ["HF_TOKEN"],
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print(model.print_trainable_parameters())  # should say ~4M trainable params

## Cell 4 — Training Pipeline (imported from train_grpo.py)

Dataset construction, drift scheduling, and reward scoring all live in
`training/train_grpo.py` (Phase 3.4) — this notebook imports them rather than
redefining them, so it can't drift out of sync with the script the way the old
notebook's hand-duplicated prompt/reward code did (F7).

In [ ]:
from raaga_env.prompting import Arm
from training.train_grpo import (
    EPISODE_LENGTH,
    build_dataset,
    make_step_reward_fn,
    sample_drift_schedule,
)

# Which observation arm to train against. HIDDEN is the actual research
# question (updatedplan.md Phase 4.1) and the one worth training first if
# you're short on time — it's the arm nothing else in this project's history
# has ever measured. DIAL and ORACLE are comparison points, not required for
# a first working model. To train another arm, change this line and re-run
# from here down (Cells 4-8); RUN_NAME in Cell 6 picks up the arm name
# automatically so runs don't overwrite each other on Drive/wandb.
ARM = Arm.HIDDEN

In [ ]:
# GRPO scores one sampled action per completion: restore the env to the row's
# exact snapshot state (Phase 0.4), apply the action, then continue with a
# reference policy for a short horizon (Monte-Carlo return) — never a
# shared/mutated env instance across samples (Phase 3.1), never an unrelated
# freshly-reset episode (F1).
reward_fn = make_step_reward_fn(ARM)

## Cell 5 — Build Dataset

Each row is one decision point: walk a fresh reference episode forward under a
random-valid reference policy (with a sampled drift schedule — Phase 3.3:
drift_at_step uniform in [16, 48], flipped to the opposite raga, 20% no-switch
control condition), stop at a uniformly chosen step, and snapshot the prompt
(with `StepFeedback` from the immediately preceding step) plus the exact env
state. No HTTP (Phase 3.1).

In [ ]:
STEPS = 300   # on T4: ~45-60 min. Increase if you have A100 credits.
BATCH = 4

print("Building training dataset...")
train_dataset = build_dataset(STEPS * BATCH, ARM)
print(f"Dataset size: {len(train_dataset)} decision points")
n_with_feedback = sum(1 for row in train_dataset if "Last action" in row["prompt"])
print(f"Rows with feedback (i.e. not step 0 of their episode): {n_with_feedback}/{len(train_dataset)}")
print(f"Sample prompt:\n{train_dataset[0]['prompt']}")

## Cell 6 — Training Config + Run

In [ ]:
import wandb
from trl import GRPOConfig, GRPOTrainer

RUN_NAME = f"jugalbandi-grpo-{ARM.value}-v1"

wandb.init(project="jugalbandi", name=RUN_NAME, config={"arm": ARM.value, "steps": STEPS, "batch": BATCH})

config = GRPOConfig(
    output_dir=f"/content/drive/MyDrive/jugalbandi/{RUN_NAME}",
    num_train_epochs=1,
    max_steps=STEPS,
    # per_device_train_batch_size * num_processes must be divisible by
    # num_generations (TRL GRPOTrainer constraint) — 2 % 4 != 0 would crash.
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    logging_steps=10,
    save_steps=50,
    report_to="wandb",
    num_generations=4,
    max_completion_length=8,  # one action index; 1-2 tokens
)

trainer = GRPOTrainer(
    model=model,
    args=config,
    reward_funcs=reward_fn,
    train_dataset=train_dataset,
    processing_class=tokenizer,        # `tokenizer=` is deprecated in TRL 0.12+
)

print(f"Starting training: {STEPS} steps, arm={ARM.value}")
trainer.train()
print("Training complete.")

**What to watch on wandb:**
- `train/reward_mean` should climb over 200+ steps
- `train/reward_std` collapsing to 0 means the model converged (possibly degenerate — check `safe_set_occupancy`-style behavior once Phase 4's eval harness is in place)
- Per-step rewards here are Monte-Carlo returns (immediate + a short reference-policy continuation), not single-step rewards — noisier than a pure bandit signal but far less noisy than scoring a full blind 64-action episode
- `grpo-HIDDEN` may simply fail to separate from a random-valid baseline at 0.5B — the risk register in updatedplan.md names this explicitly as a *possible, still publishable* result, not a bug, provided Phase 4's baselines (random-valid, safe-set-cycle, scripted-oracle) bracket it

## Cell 7 — Save and Push to Hub

In [ ]:
from huggingface_hub import HfApi

SAVE_DIR = f"/content/drive/MyDrive/jugalbandi/{RUN_NAME}/final"

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved to {SAVE_DIR}")

# CHANGE ME if jayyyyqwq isn't your Hugging Face username (GitHub handle and
# HF username don't have to match) — check huggingface.co/settings/profile.
# Push is optional: the adapter is already safe on your Drive at SAVE_DIR
# above even if you skip this cell.
api = HfApi()
api.upload_folder(
    folder_path=SAVE_DIR,
    repo_id="jayyyyqwq/jugalbandi-qwen-grpo",
    repo_type="model",
    token=os.environ["HF_TOKEN"],
    private=True,  # keep it off the public Hub index; drop this arg if you want it public later
)
print("Pushed to HuggingFace Hub.")

## Cell 8 — Inference Check

Verify the model actually learned something before claiming results. This runs
a genuine multi-turn interactive episode: one model call per step, each prompt
showing the live `StepFeedback` from the step before — exactly the interaction
pattern the model was trained for (Phase 2.3 / Claim B), and the same pattern
`openenv_server/server.py`'s `/infer` + `/step` loop uses in production.

In [ ]:
from unsloth import FastLanguageModel
from raaga_env.jugalbandi_env import JugalbandiEnv
from raaga_env.prompting import StepFeedback, note_name, parse_action, render_prompt

model, tokenizer = FastLanguageModel.from_pretrained(SAVE_DIR, load_in_4bit=True)
FastLanguageModel.for_inference(model)

initial_dial, schedule = sample_drift_schedule()
schedule_by_step = schedule.as_dict()
env = JugalbandiEnv(initial_dial=initial_dial, episode_length=EPISODE_LENGTH)
obs, info = env.reset()

feedback = None
total_reward = 0.0
forbidden_count = 0

for step_idx in range(EPISODE_LENGTH):
    if step_idx in schedule_by_step:
        env.set_dial(schedule_by_step[step_idx])
        obs = env._get_obs()

    raga = env.drift.active_raga_name if ARM is Arm.ORACLE else None
    prompt = render_prompt(
        list(obs), arm=ARM, tala_pos=env.tala_position, feedback=feedback, raga=raga,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    output = model.generate(**inputs, max_new_tokens=8, temperature=0.1)
    decoded = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    action = parse_action(decoded)
    if action is None:
        print(f"Step {step_idx}: unparseable/out-of-range output {decoded!r} — stopping early.")
        break

    note, duration = env._decode(action)
    obs, reward, terminated, truncated, info = env.step(action)
    feedback = StepFeedback(note_name=note_name(note, duration), reward=float(reward))
    total_reward += reward
    if "forbidden_note" in info.get("reward_breakdown", {}):
        forbidden_count += 1
    if terminated or truncated:
        break

switch_step = schedule.switches[0][0] if schedule.switches else None
print(f"Drift schedule: initial={initial_dial}, switch_step={switch_step}")
print(f"Steps completed: {env.step_count}/{EPISODE_LENGTH}")
print(f"Episode return: {total_reward:.2f}")
print(f"Forbidden-note steps: {forbidden_count}/{env.step_count}")